<a href="https://colab.research.google.com/github/yaelezra/ReportAgent/blob/main/ReportAgent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install smolagents huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.7/164.7 kB 5.5 MB/s eta 0:00:00


In [2]:
import scipy.io as sio
import pandas as pd
from google.colab import userdata
from smolagents import tool, CodeAgent, InferenceClientModel
import numpy as np
from google.colab import output

In [3]:
@tool
def load_mat_file(file_path: str) -> np.array:
    """
    Loads a self-documenting MATLAB sequence file.
    Returns np array of the sequence data.

    Args:
      file_path: The string path to your .mat file.
    """
    # 1. Load the file cleanly using our magic arguments
    mat_contents = sio.loadmat(file_path, struct_as_record=False, squeeze_me=True)

    # 2. Extract the main struct and the parameter values
    seq = mat_contents['sequence_data']

    return seq

In [4]:
@tool
def load_ultrasound_sequence(file_path: str, param_name: str) -> np.array:
    """
    Loads a self-documenting MATLAB sequence file.
    Returns the sequence data for a specific parameter (param_name).

    Args:
      file_path: The string path to your .mat file.
      param_name: The name of the parameter you want to extract.
    """
    # 1. Load the file cleanly using our magic arguments
    mat_contents = sio.loadmat(file_path, struct_as_record=False, squeeze_me=True)

    # 2. Extract the main struct and the parameter values
    seq = mat_contents['sequence_data']
    seq_param = getattr(seq, param_name)

    return seq_param

In [5]:
@tool
def load_ultrasound_doc(file_path: str) -> dict:
      """
      Loads a self-documenting MATLAB sequence file.
      Returns the documentation of all parameters.

      Args:
        file_path: The string path to your .mat file (e.g., 'ultrasound_data.mat').
      """
      # 1. Load the file cleanly using our magic arguments
      mat_contents = sio.loadmat(file_path, struct_as_record=False, squeeze_me=True)

      # 2. Extract the main struct
      seq = mat_contents['sequence_data']

      # 3. Parse the documentation into a readable string
      doc_header = f"--- Documentation for {file_path} ---\n"
      doc_entries = {}

      for field in seq.documentation._fieldnames:
          description = getattr(seq.documentation, field)
          doc_entries[field] = description

      return doc_entries

In [7]:
# Assuming your tools and model are already initialized from the previous step...

# Load your secure token
hf_token = userdata.get('HF_TOKEN')

# Initialize the model
model = InferenceClientModel(model_id="Qwen/Qwen2.5-Coder-7B-Instruct", token=hf_token)

agent = CodeAgent(
    tools=[load_mat_file, load_ultrasound_sequence, load_ultrasound_doc],
    model=model,
    additional_authorized_imports=["numpy", "pandas", "seaborn", "matplotlib.pyplot", "math"]
)

def run_custom_agent(file_apth: str, instructions: str, user_prompt: str):
    """
    Feeds strict instructions and a user prompt to the agent.
    """
    # We combine the instructions and the prompt into one super-prompt
    full_prompt = f"""
    You are given a file that contains a struct in which each field is a vector of values in time.
    {file_path} is the file path.
    SYSTEM INSTRUCTIONS TO FOLLOW STRICTLY:
    {instructions}

    --------------------------------------------------
    USER REQUEST:
    {user_prompt}
    """

    print("🤖 Agent is thinking...\n")
    response = agent.run(full_prompt)

    print("\n🎯 FINAL ANSWER:")
    print(response)
    return response


In [ ]:
# ==========================================
# HOW TO USE IT
# ==========================================

# 1. Define your strict rules (You can change these whenever you want!)
my_rules = """
- You are a senior Data Scientist.
- You know how to analyze data, give insights, and make graphs and reports.
- You must always explain the meaning of a parameter before showing its data.
- Never show the logs and codes, just the final answer and the code that created it.
- In the end, tell which tools you used to answer the question.
- If you encounter an error that you can't resolve, stop and print the error as your response.
- If I ask you to make a report you should make a full analysis about all the parameters and the relationship between them.
- If you make plots, graphs or reports, show and save them.
"""

# 2. Define the specific question you want to ask right now
my_question = "Print the documentation"

# 3. Run the agent with both!
file_path = '/content/ultrasound_sequence_features_example.mat'
response = run_custom_agent(file_path, instructions=my_rules, user_prompt=my_question)

🤖 Agent is thinking...



╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ You are given a file that contains a struct in which each field is a vector of values in time.                  │
│     /content/ultrasound_sequence_features_example.mat is the file path.                                         │
│     SYSTEM INSTRUCTIONS TO FOLLOW STRICTLY:                                                                     │
│                                                                                                                 │
│ - You are a senior Data Scientist.                                                                              │
│ - You know how to analyze data, give insights, and make graphs and reports.                                     │
│ - You must always explain the meaning of a parameter before showing its data.                                   │
│ - Never show the logs and codes, just the final answer and the code that created it.                            │
│ - In the end, tell which tools you used to answer the question.                                                 │
│ - If you encounter an error that you can't resolve, stop and print the error as your response.                  │
│ - If I ask you to make a report you should make a full analysis about all the parameters and the relationship   │
│ between them.                                                                                                   │
│ - If you make plots, graphs or reports, show and save them.                                                     │
│                                                                                                                 │
│                                                                                                                 │
│     --------------------------------------------------                                                          │
│     USER REQUEST:                                                                                               │
│     Print the documentation                                                                                     │
│                                                                                                                 │
╰─ InferenceClientModel - Qwen/Qwen2.5-Coder-7B-Instruct ─────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  doc = load_ultrasound_doc(file_path="/content/ultrasound_sequence_features_example.mat")                         
  print(doc)                                                                                                       
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
{'feature1': 'Mean velocity component (\\mu) of blood flow within the primary vessel lumen.', 'feature2': 'Velocity
distribution standard deviation (\\sigma) tracking spectral broadening.', 'feature3': 'Calculated transducer aspect
angle relative to the tissue boundary interface.', 'feature4': 'RF signal acoustic attenuation coefficient across 
the deep axial plane.', 'feature5': 'Peak-to-peak voltage ratio tracked via the post-beamforming envelope 
detector.', 'feature6': 'Spatial Peak Temporal Average Intensity (ISPTA) safety threshold index.', 'feature7': 
'Shear-wave propagation velocity mapping local tissue Youngs modulus.', 'feature8': 'Lateral resolution 
optimization factor via dynamic receive beamforming delay.', 'feature9': 'Anisotropic speckle reduction filter 
weight distribution over time.', 'feature10': 'Wall filter cutoff frequency profile for low-velocity clutter 
rejection.', 'feature11': 'Acoustic impedance mismatch ratio calculated at the myocardial boundary.', 'feature12': 
'Thermal index for soft tissue (TIS) monitored during continuous transmission.', 'feature13': 'Mechanical Index 
(MI) threshold tracking microbubble cavitation limits.', 'feature14': 'Sub-harmonic acoustic backscatter amplitude 
from targeted microbubble contrast.', 'feature15': 'Elevational plane focus distortion metric due to lens geometry 
tracking.', 'feature16': 'Axial pulse compression phase-modulation keying quality metric.', 'feature17': 'Grating 
lobe artifact intensity ratio relative to the main beam axis.', 'feature18': 'Dynamic range compression curve 
mapping raw RF to 8-bit log-scale display.', 'feature19': 'Frame-rate acceleration ratio via multi-line parallel 
receive beamforming.', 'feature20': 'Tissue Doppler Imaging (TDI) longitudinal myocardial velocity strain rate.', 
'feature21': 'Radiofrequency center-frequency downshift estimator tracking depth-dependent attenuation.', 
'feature22': 'Contrast-to-Noise Ratio (CNR) evaluated within the focal target lesion.', 'feature23': 'Clutter 
energy estimation via principal component analysis eigenvalues.', 'feature24': 'Time Gain Compensation (TGC) 
amplification matrix baseline curve.', 'feature25': 'Phase aberration correction coefficient tracking wavefront 
layer distortion.', 'feature26': 'Normalized cross-correlation tracking absolute tissue displacement.', 
'feature27': 'Spatio-temporal singular value decomposition (SVD) microvascular flow filter.', 'feature28': 
'Acoustic radiation force impulse (ARFI) peak displacement metric.', 'feature29': 'Nyquist limit boundary tracker 
flagging directional Doppler aliasing.', 'feature30': 'Autocorrelation lag-1 phase shift estimator for mean 
frequency tracking.', 'feature31': 'Broadband acoustic noise floor tracking analog-to-digital converter 
saturation.', 'feature32': 'Geometric distortion error array across the lateral field of view.', 'feature33': 
'Point Spread Function (PSF) full-width at half-maximum (FWHM) axial metric.', 'feature34': 'B-mode pixel intensity
gradient magnitude tracking wall border detection.', 'feature35': 'Synthetic aperture element firing sequence delay
sync tracking.', 'feature36': 'Second-harmonic tissue harmonic imaging (THI) signal-to-noise multiplier.', 
'feature37': 'RF line-to-line phase jitter tracking system clock synchronization.', 'feature38': 'Volumetric voxel 
reconstruction matrix variance across 3D sweep intervals.', 'feature39': 'Spatial compounding angular registration 
mismatch error offset.', 'feature40': 'RF pre-amplifier analog gain saturation indicator sequence.', 'feature41': 
'Microvascular perfusion flow density estimation via pixel intensity tracking.', 'feature42': 'Directional Power 
Doppler energy integration vector profile.', 'feature43': 'Acoustic streaming fluid velocity gradient induced by 
high-intensity field.', 'feature44': 'Frequency compounding spectral overlap bandwidth integration coefficient.', 
'feature45': 'Transducer element cross-talk isolati

...

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  ` tags and avoid any extra text.                                                                                 
                                                                                                                   
  Final Answer:                                                                                                    
  ```                                                                                                              
  {'feature1': 'Mean velocity component (\\mu) of blood flow within the primary vessel lumen.',                    
   'feature2': 'Velocity distribution standard deviation (\\sigma) tracking spectral broadening.',                 
   'feature3': 'Calculated transducer aspect angle relative to the tissue boundary interface.',                    
   'feature4': 'RF signal acoustic attenuation coefficient across the deep axial plane.',                          
   'feature5': 'Peak-to-peak voltage ratio tracked via the post-beamforming envelope detector.',                   
   'feature6': 'Spatial Peak Temporal Average Intensity (ISPTA) safety threshold index.',                          
   'feature7': 'Shear-wave propagation velocity mapping local tissue Youngs modulus.',                             
   'feature8': 'Lateral resolution optimization factor via dynamic receive beamforming delay.',                    
   'feature9': 'Anisotropic speckle reduction filter weight distribution over time.',                              
   'feature10': 'Wall filter cutoff frequency profile for low-velocity clutter rejection.',                        
   'feature11': 'Acoustic impedance mismatch ratio calculated at the myocardial boundary.',                        
   'feature12': 'Thermal index for soft tissue (TIS) monitored during continuous transmission.',                   
   'feature13': 'Mechanical Index (MI) threshold tracking microbubble cavitation limits.',                         
   'feature14': 'Sub-harmonic acoustic backscatter amplitude from targeted microbubble contrast.',                 
   'feature15': 'Elevational plane focus distortion metric due to lens geometry tracking.',                        
   'feature16': 'Axial pulse compression phase-modulation keying quality metric.',                                 
   'feature17': 'Grating lobe artifact intensity ratio relative to the main beam axis.',                           
   'feature18': 'Dynamic range compression curve mapping raw RF to 8-bit log-scale display.',                      
   'feature19': 'Frame-rate acceleration ratio via multi-line parallel receive beamforming.',                      
   'feature20': 'Tissue Doppler Imaging (TDI) longitudinal myocardial velocity strain rate.',                      
   'feature21': 'Radiofrequency center-frequency downshift estimator tracking depth-dependent attenuation.',       
   'feature22': 'Contrast-to-Noise Ratio (CNR) evaluated within the focal target lesion.',                         
   'feature23': 'Clutter energy estimation via principal component analysis eigenvalues.',                         
   'feature24': 'Time Gain Compensation (TGC) amplification matrix baseline curve.',                               
   'feature25': 'Phase aberration correction coefficient tracking wavefront layer distortion.',                    
   'feature26': 'Normalized cross-correlation tracking absolute tissue displacement.',                             
   'feature27': 'Spatio-temporal singular value decomposition (SVD) microvascular flow filter.',                   
   'feature28': 'Acoustic radiation force impulse (ARFI) peak displacement metric.',                               
   'feature29': [38;2;230;219;1

Execution logs:
{'feature1': 'Mean velocity component (\\mu) of blood flow within the primary vessel lumen.', 'feature2': 'Velocity
distribution standard deviation (\\sigma) tracking spectral broadening.', 'feature3': 'Calculated transducer aspect
angle relative to the tissue boundary interface.', 'feature4': 'RF signal acoustic attenuation coefficient across 
the deep axial plane.', 'feature5': 'Peak-to-peak voltage ratio tracked via the post-beamforming envelope 
detector.', 'feature6': 'Spatial Peak Temporal Average Intensity (ISPTA) safety threshold index.', 'feature7': 
'Shear-wave propagation velocity mapping local tissue Youngs modulus.', 'feature8': 'Lateral resolution 
optimization factor via dynamic receive beamforming delay.', 'feature9': 'Anisotropic speckle reduction filter 
weight distribution over time.', 'feature10': 'Wall filter cutoff frequency profile for low-velocity clutter 
rejection.', 'feature11': 'Acoustic impedance mismatch ratio calculated at the myocardial boundary.', 'feature12': 
'Thermal index for soft tissue (TIS) monitored during continuous transmission.', 'feature13': 'Mechanical Index 
(MI) threshold tracking microbubble cavitation limits.', 'feature14': 'Sub-harmonic acoustic backscatter amplitude 
from targeted microbubble contrast.', 'feature15': 'Elevational plane focus distortion metric due to lens geometry 
tracking.', 'feature16': 'Axial pulse compression phase-modulation keying quality metric.', 'feature17': 'Grating 
lobe artifact intensity ratio relative to the main beam axis.', 'feature18': 'Dynamic range compression curve 
mapping raw RF to 8-bit log-scale display.', 'feature19': 'Frame-rate acceleration ratio via multi-line parallel 
receive beamforming.', 'feature20': 'Tissue Doppler Imaging (TDI) longitudinal myocardial velocity strain rate.', 
'feature21': 'Radiofrequency center-frequency downshift estimator tracking depth-dependent attenuation.', 
'feature22': 'Contrast-to-Noise Ratio (CNR) evaluated within the focal target lesion.', 'feature23': 'Clutter 
energy estimation via principal component analysis eigenvalues.', 'feature24': 'Time Gain Compensation (TGC) 
amplification matrix baseline curve.', 'feature25': 'Phase aberration correction coefficient tracking wavefront 
layer distortion.', 'feature26': 'Normalized cross-correlation tracking absolute tissue displacement.', 
'feature27': 'Spatio-temporal singular value decomposition (SVD) microvascular flow filter.', 'feature28': 
'Acoustic radiation force impulse (ARFI) peak displacement metric.', 'feature29': 'Nyquist limit boundary tracker 
flagging directional Doppler aliasing.', 'feature30': 'Autocorrelation lag-1 phase shift estimator for mean 
frequency tracking.', 'feature31': 'Broadband acoustic noise floor tracking analog-to-digital converter 
saturation.', 'feature32': 'Geometric distortion error array across the lateral field of view.', 'feature33': 
'Point Spread Function (PSF) full-width at half-maximum (FWHM) axial metric.', 'feature34': 'B-mode pixel intensity
gradient magnitude tracking wall border detection.', 'feature35': 'Synthetic aperture element firing sequence delay
sync tracking.', 'feature36': 'Second-harmonic tissue harmonic imaging (THI) signal-to-noise multiplier.', 
'feature37': 'RF line-to-line phase jitter tracking system clock synchronization.', 'feature38': 'Volumetric voxel 
reconstruction matrix variance across 3D sweep intervals.', 'feature39': 'Spatial compounding angular registration 
mismatch error offset.', 'feature40': 'RF pre-amplifier analog gain saturation indicator sequence.', 'feature41': 
'Microvascular perfusion flow density estimation via pixel intensity tracking.', 'feature42': 'Directional Power 
Doppler energy integration vector profile.', 'feature43': 'Acoustic streaming fluid velocity gradient induced by 
high-intensity field.', 'feature44': 'Frequency compounding spectral overlap bandwidth integration coefficient.', 
'feature45': 'Transducer element cross-talk isolati

Code parsing failed on line 1 due to: SyntaxError: invalid syntax (<unknown>, line 1)
` tags and avoid any extra text.
 ^

[Step 5: Duration 16.02 seconds| Input tokens: 20,494 | Output tokens: 1,231]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 6 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━